In [ ]:
import pandas as pd
import re
import os
import librosa
import soundfile as sf
import numpy as np
import random

In [ ]:
# Lsit folder
input_folder = '../raw_mjf_tracks'
mfj_tracks = os.listdir(input_folder) 

# Create dataframe
mfj_df = pd.DataFrame(columns=['Artist','Title','Year', 'Genre','Path',
                               'Wonder', 'Transcendence', 'Nostalgia',
                               'Tenderness', 'Peacefulness', 'Joy',
                               'Power','Tension','Sadness'])


In [ ]:
def extract_random_excerpt(input_path: str, output_path: str, output_folder = 'mjf_tracks', sr = 16000, duration = 45):
    '''
    Extract a random excerpt from audio.
    '''

    # load the audio
    y, _ = librosa.load(input_path,
                     mono=True,
                     sr=sr)
    
    total_samples = y.shape[-1]
    total_duration = total_samples / sr

    # Selects a random excerpt if the audio is longer than the decided excerpt duration
    if total_duration > duration:

        max_start = total_duration - duration
        start_time = random.uniform(0, max_start)
        start_sample = int(start_time * sr)
        end_sample = start_sample + int(duration * sr)

        excerpt = y[start_sample:end_sample]
    else :
        excerpt = y

    # Creates output directory
    os.makedirs(output_folder, exist_ok=True)

    # Saves the excerpt
    sf.write(f'{output_folder}/{output_path}', excerpt, sr)
    print(f'{output_folder}/{output_path}')
    

    

In [ ]:
for track in mfj_tracks:

    # Path to the audio
    audio_file = f'../raw_mjf_tracks/{track}'

    # Regex to match artist, title and year in the filename
    groups = re.findall( r'(.+?)(\s-\s)', track)
    year = re.findall(r'\d{2}(?=[A-Z]{4})', track)[0]
    year = f'20{year}' if int(year) < 26 else f'19{year}' # complete year
    
    # Title is not always in the filename
    artist = groups[0][0]
    title = groups[1][0] if len(groups) > 1 else ''

    id = len(mfj_df)
    output_path = f'{id}_{artist}-{title}-{year}.wav'
    
    try:
        extract_random_excerpt(input_path=audio_file,
                           output_path=output_path)
    except:
        print(f'Couldn\'t extract : {output_path}' )
    
    # Add infos tho the dataframe
    infos = {'Artist': artist, 'Title': title, 'Year': year, 'Path' : output_path}
    mfj_df.loc[id] = infos

In [ ]:
# Saves the dataframe
mfj_df.to_csv('mjf_tracks.csv')